# Validating COMPASS on three OpenNeuro datasets

This notebook runs the COMPASS multi-agent inference engine end to end on three open clinical
datasets and checks how well it recovers each dataset's clinical phenotype. It loads the pre-built,
blinded COMPASS inputs, asserts every record is well formed, runs a SUBSET (2 subjects per dataset on a
substantial representative tier) under a spend guard, visualizes the recovered phenotypes against ground
truth, and estimates the API cost and wall time of the FULL batch (the whole cohort across every tier).

Concurrency note: each engine run already fans out several tool/LLM calls internally
(`COMPASS_EXECUTOR_MAX_WORKERS`, default 12). Running many engine instances in an outer pool multiplies
the concurrent OpenRouter requests (outer x inner), which exhausts the connection pool / rate limit and
stalls (the provider rate-limits above about 3 to 4 concurrent large-context calls), so we run one
participant at a time (outer pool = 1) with the executor fan-out bounded to 3, keeping the total near 3
concurrent calls. Each run does deep multi-step reasoning, so it takes a few minutes; the whole subset takes on the
order of 20 to 30 minutes. The very largest tiers (the full 836-feature EEG and the brain connectome)
build huge prompts that the flash model reasons over slowly, so the subset uses a substantial tier that
completes reliably, while the full ladder (including those big tiers) is quantified in the cost estimate.

| Dataset | Accession | Clinical phenotype (prediction output structure) | Subset tier -> full tier |
|---|---|---|---|
| INTELLIGENCE (AOMIC-ID1000) | ds003097 | total intelligence (univariate) then 3 IST subscales (multivariate) | T4 all self-report -> T6 + brain |
| PSYCHOSIS (first-episode) | ds003944 + ds003947 | diagnosis (binary) then BPRS total (univariate) then SAPS/SANS globals (multivariate) | T2 clinical profile -> T3 + 836 EEG |
| NUMERACY (stroke) | ds006533 | approximate and precise numeracy (two dissociable univariate phenotypes) | T3 per-parcel lesion overlap (full) |

Each dataset's exact tiers and phenotype structure are documented in its `PHENOTYPE_AND_TIERS.md`.
Model: **deepseek/deepseek-v4-flash** via OpenRouter. **How to use:** run the SETUP, REGISTRY and
LOAD-CHECK cells once, then run each dataset's own SUBSET cell and its VISUALIZE cell (they are separate
so you can run them independently in PyCharm/Jupyter). The FULL-BATCH cells at the end are guarded and are
NOT run here; the cost-estimation cell tells you what they would cost first.

No target value (IST/BPRS/SAPS/SANS/numeracy) is ever an input: every phenotype is inferred only from the
multimodal predictor evidence, with native scales described to the engine in each record.

In [ ]:
import os, sys, json, io, time, shutil, contextlib, warnings, tempfile, urllib.request, importlib.util
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
warnings.filterwarnings("ignore")
# The engine parallelizes each run internally (its plan executor fans out up to COMPASS_EXECUTOR_MAX_WORKERS
# tool calls at once, embeddings up to COMPASS_EMBEDDING_MAX_WORKERS). On large tiers, 12 concurrent
# large-context calls exhaust the connection pool / hit the OpenRouter rate limit and stall, so we bound
# the internal fan-out and run one participant at a time (outer pool = 1): total concurrent provider calls
# stays near 3, which runs smoothly. For the full batch, raise these only as far as your rate limit allows.
os.environ.setdefault("COMPASS_EXECUTOR_MAX_WORKERS", "3")
os.environ.setdefault("COMPASS_EMBEDDING_MAX_WORKERS", "8")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
try:  # render figures inline when in a notebook; harmless no-op as a plain script
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

# Repo root = the folder containing src/full_stack (works from any working directory / PyCharm).
REPO = Path.cwd().resolve()
while not (REPO / "src" / "full_stack").is_dir():
    if REPO == REPO.parent:
        raise RuntimeError("Could not locate repo root (src/full_stack) above the working directory")
    REPO = REPO.parent
for p in (str(REPO), str(REPO / "validation")):
    if p not in sys.path:
        sys.path.insert(0, p)
VAL = REPO / "validation" / "datasets"

# All engine scratch (input copies, outputs, logs, cached results) lives in the SYSTEM temp dir, never
# inside the repo, so nothing here is ever committed.
SCRATCH = Path(tempfile.gettempdir()) / "compass_openneuro_validation"
(SCRATCH / "outputs").mkdir(parents=True, exist_ok=True)
(SCRATCH / "logs").mkdir(parents=True, exist_ok=True)

MODEL = "deepseek/deepseek-v4-flash"          # cheap, 1M-context; set once, shared by all agent roles
N_PER_DATASET = 2                              # subjects per dataset for the subset run
MAX_WORKERS = 1                                # OUTER engine runs: sequential. The provider rate-limits above
                                               # ~3-4 concurrent large-context calls, so one run at a time (each still
                                               # fans out COMPASS_EXECUTOR_MAX_WORKERS=3 internally) is the reliable setting.
MAX_USD = 8.0                                  # hard spend guard across a run
MAX_ITERATIONS = 1                             # actor-critic depth per run

from src.full_stack.backend.config.settings import LLMBackend, get_settings
S = get_settings()
S.models.backend = LLMBackend.OPENROUTER
S.models.public_model_name = MODEL
for role in ("orchestrator", "critic", "predictor", "integrator", "communicator", "tool"):
    setattr(S.models, f"{role}_model", MODEL)
S.paths.output_dir = SCRATCH / "outputs"
S.paths.logs_dir = SCRATCH / "logs"

import ssl
try:
    import certifi
    _SSL_CTX = ssl.create_default_context(cafile=certifi.where())   # Framework Python lacks system CAs
except Exception:
    _SSL_CTX = ssl.create_default_context()

def openrouter_usage():
    "Total OpenRouter USD used so far (None if unavailable)."
    key = S.openrouter_api_key
    if not key:
        return None
    try:
        req = urllib.request.Request("https://openrouter.ai/api/v1/credits",
                                     headers={"Authorization": f"Bearer {key}"})
        with urllib.request.urlopen(req, timeout=15, context=_SSL_CTX) as r:
            return float(json.loads(r.read().decode()).get("data", {}).get("total_usage", 0.0))
    except Exception:
        return None

assert S.openrouter_api_key, "OPENROUTER_API_KEY missing from repo-root .env"
_u = openrouter_usage()
print(f"repo:    {REPO}")
print(f"scratch: {SCRATCH}  (system temp, never committed)")
print(f"model:   {MODEL}  |  workers: {MAX_WORKERS}  |  spend cap per run: +${MAX_USD:.2f}")
print(f"OpenRouter usage so far: {'$'+format(_u,'.4f') if _u is not None else 'unavailable'}")

## 1. Dataset registry

Each dataset contributes one or more prediction TASKS (psychosis and intelligence each have a single
hierarchical task; numeracy has two univariate tasks, one per numeracy system). For every dataset we hold
its full tier ladder, the full-complexity tier used for the subset run, the two chosen subjects, and each
task's `PredictionTaskSpec` + global instruction (native scales, imported from the dataset's own pipeline
module so the engine stays dataset-agnostic). Ground truth is loaded for scoring only, never shown to the
engine.

In [ ]:
def _load(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); sys.modules[name] = m
    spec.loader.exec_module(m); return m

from src.full_stack.backend.data.models.prediction_task import (
    PredictionMode, PredictionTaskNode, PredictionTaskSpec)

DATASETS = {}   # key -> dict(label, tiers, full_tier, tasks, subjects, n_cohort, src_dir, ground_truth, diagnosis)

# ---------------- INTELLIGENCE (AOMIC-ID1000) ----------------
ist = _load(VAL / "INTELLIGENCE" / "pipeline" / "config.py", "ist_config")
ist_ann = {a["participant_id"]: a for a in
           json.load(open(VAL / "INTELLIGENCE" / "results" / "annotations.json"))["annotations"]}
_part = pd.read_csv(VAL / "INTELLIGENCE" / "dataset" / "participants.tsv", sep="\t", na_values=["n/a", "N/A", ""])
_ist_outputs = [ist.TARGET["column"]] + [s["column"] for s in ist.SUBSCALES]
_ist_stats = {c: {"mean": round(float(pd.to_numeric(_part[c], errors="coerce").mean()), 2),
                  "sd": round(float(pd.to_numeric(_part[c], errors="coerce").std(ddof=0)), 2)} for c in _ist_outputs}

def _ist_spec():
    return PredictionTaskSpec(task_id="ist_hierarchical", root=PredictionTaskNode(
        node_id="total_intelligence", display_name=ist.TARGET["label"],
        mode=PredictionMode.UNIVARIATE_REGRESSION, regression_outputs=[ist.TARGET["column"]],
        unit_by_output={ist.TARGET["column"]: "IST points"},
        children=[PredictionTaskNode(node_id="ist_subscales",
            display_name="IST subscales: fluid, memory, crystallised",
            mode=PredictionMode.MULTIVARIATE_REGRESSION, regression_outputs=[s["column"] for s in ist.SUBSCALES],
            unit_by_output={s["column"]: "IST points" for s in ist.SUBSCALES})]))

def _ist_gi():
    L = [ist.IST_CONTEXT, "", "Predict these related scores on their NATIVE IST scales:"]
    t = _ist_stats[ist.TARGET["column"]]
    L.append(f"- {ist.TARGET['column']} ({ist.TARGET['label']}): overall composite. "
             f"Population reference mean={t['mean']}, sd={t['sd']} native IST points.")
    for s in ist.SUBSCALES:
        st = _ist_stats[s["column"]]
        L.append(f"- {s['column']} ({s['label']}): {s['description']} mean={st['mean']}, sd={st['sd']}.")
    L += ["", "Subscales are components of the total and move with it. No IST value for this participant is "
          "provided; infer every score only from the non-cognitive multimodal evidence. Return one numeric "
          "value per output on its native scale."]
    return "\n".join(L)

_ist_sorted = sorted(ist_ann.values(), key=lambda a: a["ground_truth"][ist.TARGET["column"]])
DATASETS["INTELLIGENCE"] = dict(
    label="AOMIC-ID1000 (ds003097): intelligence",
    tiers=["T1_demographics", "T2_personality", "T3_psychometric", "T4_identity", "T5_morphometry", "T6_connectome"],
    full_tier="T6_connectome", subset_tier="T4_identity", n_cohort=len(ist_ann),
    tasks=[dict(name="ist", spec=_ist_spec(), gi=_ist_gi(), outputs=_ist_outputs,
                target_label=ist.TARGET["label"], control_label="")],
    subjects=[_ist_sorted[0]["participant_id"], _ist_sorted[-1]["participant_id"]][:N_PER_DATASET],
    src_dir=lambda tier, subj, task: VAL / "INTELLIGENCE" / "compass_inputs" / tier / subj,
    ground_truth=lambda subj, task: ist_ann[subj]["ground_truth"],
    diagnosis=lambda subj: None)

# ---------------- PSYCHOSIS (first-episode) ----------------
sys.path.insert(0, str(VAL / "PSYCHOSIS_FIRST_EPISODE"))
from utils import compass_task as PSY
_psy_ec = json.load(open(VAL / "PSYCHOSIS_FIRST_EPISODE" / "results" / "compass" / "eval_cohort.json"))
_psy_annfile = json.load(open(VAL / "PSYCHOSIS_FIRST_EPISODE" / "results" / "compass" / "annotations.json"))
_psy_ann = {a["recording_id"]: a for a in _psy_annfile["annotations"]}
DATASETS["PSYCHOSIS"] = dict(
    label="First-episode psychosis (ds003944+ds003947)",
    tiers=["T1_demographic_socioeconomic", "T2_clinical_profile", "T3_multimodal_full", "T4_eeg_lean", "T5_eeg_rich"],
    full_tier="T3_multimodal_full", subset_tier="T2_clinical_profile",
    n_cohort=_psy_annfile["n_recordings"],  # full dataset = 143 recordings
    tasks=[dict(name="psy", spec=PSY.build_task_spec(), gi=_psy_ec["global_instruction"],
                outputs=PSY.ALL_OUTPUTS, target_label=PSY.CASE_LABEL, control_label=PSY.CONTROL_LABEL)],
    subjects=[_psy_ec["psychosis"][0], _psy_ec["control"][0]][:N_PER_DATASET],  # 1 case + 1 control
    src_dir=lambda tier, subj, task: VAL / "PSYCHOSIS_FIRST_EPISODE" / "results" / "compass" / "inputs" / tier / subj,
    ground_truth=lambda subj, task: _psy_ann[subj]["ground_truth"],
    diagnosis=lambda subj: _psy_ann[subj]["diagnosis"])

# ---------------- NUMERACY (stroke) ----------------
num = _load(VAL / "NUMERACY_STROKE" / "pipeline" / "compass_task.py", "num_task")
_num_tasks, _num_gt = [], {}
for _t in num.TARGETS:
    _m, _s = num.reference_stats(_t)
    _num_tasks.append(dict(name=_t, spec=num.build_task_spec(_t), gi=num.build_global_instruction(_t, _m, _s),
                           outputs=[_t], target_label=num.TARGET_LABEL[_t], control_label=""))
    sub = json.load(open(VAL / "NUMERACY_STROKE" / "results" / f"subset_{_t}.json"))
    _num_gt[_t] = {p["participant_id"]: {_t: p["ground_truth"]} for p in sub["participants"]}
_num_blind = VAL / "NUMERACY_STROKE" / "compass_inputs"
def _num_present(subj):
    return all((_num_blind / f"{num.TARGET_SHORT[t]}_{lv}_blinded" / subj).is_dir()
               for t in num.TARGETS for lv in num.LEVELS)
DATASETS["NUMERACY"] = dict(
    label="Numeracy after stroke (ds006533)",
    tiers=num.LEVELS, full_tier="T3_lesion_fine", subset_tier="T3_lesion_coarse",
    n_cohort=json.load(open(VAL / "NUMERACY_STROKE" / "results" / "annotations.json"))["n_subjects"],  # 105
    tasks=_num_tasks,
    subjects=[s for s in sorted(_num_gt[num.TARGETS[0]]) if _num_present(s)][:N_PER_DATASET],
    src_dir=lambda tier, subj, task: _num_blind / f"{num.TARGET_SHORT[task]}_{tier}_blinded" / subj,
    ground_truth=lambda subj, task: _num_gt[task][subj],
    diagnosis=lambda subj: None)

for k, d in DATASETS.items():
    n = len(d["subjects"]) * len(d["tasks"])
    print(f"{k:12} subset tier={d['subset_tier']:26} ({n} runs) | full tier={d['full_tier']:20} "
          f"| ladder={len(d['tiers'])} tiers, cohort~{d['n_cohort']} | subjects={d['subjects']}")

## 2. Engine run helpers and load-check

`run_job` copies a record's four files into a uniquely named participant directory (in system temp) so
`ThreadPoolExecutor` workers never collide on the engine's per-participant output path, runs the full
COMPASS pipeline, and harvests the prediction from the in-memory result. `run_dataset` builds the jobs for
one dataset over the requested tiers, runs them in parallel with a spend guard, and caches the results to
system temp so re-running a cell does not re-spend. The load-check then asserts every subset record parses
through the engine's own `DataLoader` with a valid ontology and no target leakage.

In [ ]:
def harvest(result):
    "Generic: pull the diagnosis label/probs and every regression value from any hierarchy."
    pred = (result.get("internal_context") or {}).get("prediction")
    root = getattr(pred, "root_prediction", None)
    out = {"label": None, "probs": None, "regression": {}}
    if root is None:
        return out
    for node in [root] + (root.walk() if hasattr(root, "walk") else []):
        cls = getattr(node, "classification", None)
        if cls is not None and out["label"] is None:
            lab = getattr(cls, "predicted_label", None) or getattr(cls, "label", None)
            pr = getattr(cls, "class_probabilities", None) or getattr(cls, "probabilities", None)
            out["label"] = None if lab is None else str(lab)
            if isinstance(pr, dict):
                out["probs"] = {str(k): float(v) for k, v in pr.items()}
        reg = getattr(node, "regression", None)
        for k, v in (getattr(reg, "values", None) or {}).items():
            try:
                out["regression"][str(k)] = float(v)
            except (TypeError, ValueError):
                pass
    return out

def build_jobs(dkey, tiers):
    d = DATASETS[dkey]; jobs = []
    for task in d["tasks"]:
        for subj in d["subjects"]:
            for tier in tiers:
                src = d["src_dir"](tier, subj, task["name"])
                jobs.append(dict(dataset=dkey, task=task["name"], tier=tier, subject=subj,
                                 src_dir=src, spec=task["spec"], gi=task["gi"], outputs=task["outputs"],
                                 target_label=task["target_label"], control_label=task["control_label"],
                                 ground_truth=d["ground_truth"](subj, task["name"]), diagnosis=d["diagnosis"](subj),
                                 uid=f"{dkey}__{task['name']}__{tier}__{subj}".replace("/", "-")))
    return jobs

def run_job(job):
    from main import run_compass_pipeline
    pdir = SCRATCH / "inputs" / job["uid"]; pdir.mkdir(parents=True, exist_ok=True)
    for f in ("data_overview.json", "hierarchical_deviation_map.json", "multimodal_data.json", "non_numerical_data.txt"):
        shutil.copy(job["src_dir"] / f, pdir / f)
    buf, t0 = io.StringIO(), time.time()
    try:
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            res = run_compass_pipeline(participant_dir=pdir, target_condition=job["target_label"],
                control_condition=job["control_label"], prediction_task_spec=job["spec"],
                agent_instructions={"global": job["gi"]}, max_iterations=MAX_ITERATIONS,
                verbose=False, interactive_ui=False)
        keep = ("dataset", "task", "tier", "subject", "outputs", "ground_truth", "diagnosis")
        return {**{k: job[k] for k in keep}, "prediction": harvest(res),
                "seconds": round(time.time() - t0, 1), "ok": True}
    except Exception as e:
        keep = ("dataset", "task", "tier", "subject", "outputs", "ground_truth", "diagnosis")
        return {**{k: job[k] for k in keep}, "error": f"{type(e).__name__}: {e}",
                "log_tail": buf.getvalue()[-800:], "ok": False}
    finally:
        shutil.rmtree(pdir, ignore_errors=True)

def _tag(r):
    if not r.get("ok"):
        return "ERROR " + r.get("error", "")[:56]
    p = r["prediction"]
    if p["label"] is not None:
        pr = max(p["probs"].values()) if p.get("probs") else None
        return f"dx={p['label'][:20]}" + (f" p={pr:.2f}" if pr else "")
    o = r["outputs"][0]; gt = (r["ground_truth"] or {}).get(o)
    return f"{o.split('__')[-1][:16]}: pred={p['regression'].get(o)}, true={gt}"

def run_dataset(dkey, tiers, rerun=False, label="subset"):
    "Run one dataset over `tiers` in parallel (spend-guarded, cached to system temp)."
    cache = SCRATCH / f"results_{dkey}_{label}.json"
    if cache.exists() and not rerun:
        rows = json.loads(cache.read_text())
        print(f"[{dkey}] loaded {len(rows)} cached {label} results (rerun=True to re-spend)")
        return rows
    jobs = build_jobs(dkey, tiers)
    u0 = openrouter_usage() or 0.0
    print(f"[{dkey}] {label}: {len(jobs)} runs on {MODEL}, {MAX_WORKERS} workers, cap +${MAX_USD:.2f}")
    ex = ThreadPoolExecutor(max_workers=MAX_WORKERS)
    it = iter(jobs); pending = {}; rows = []; stop = False
    def submit():
        try:
            j = next(it)
        except StopIteration:
            return
        pending[ex.submit(run_job, j)] = j
    for _ in range(min(MAX_WORKERS, len(jobs))):
        submit()
    done = 0
    try:
        while pending and not stop:
            fin, _ = wait(set(pending), return_when=FIRST_COMPLETED)
            for f in fin:
                pending.pop(f); done += 1; r = f.result(); rows.append(r)
                print(f"  [{done:>2}/{len(jobs)}] {r['tier']:26} {str(r['subject'])[:20]:20} "
                      f"{r.get('seconds','?')!s:>5}s  {_tag(r)}", flush=True)
                u = openrouter_usage()
                if u is not None and u - u0 >= MAX_USD:
                    print(f"  [spend guard hit +${u-u0:.2f}; stopping]"); stop = True; break
                submit()
    finally:
        for f in pending:
            f.cancel()
        ex.shutdown(wait=True)
    spent = (openrouter_usage() or u0) - u0
    cache.write_text(json.dumps(rows, indent=2, default=str))
    ok = [r for r in rows if r.get("ok")]
    (SCRATCH / f"results_{dkey}_{label}_meta.json").write_text(json.dumps({
        "spent_usd": round(spent, 5), "n_runs": len(rows), "n_ok": len(ok),
        "avg_seconds": round(float(np.mean([r["seconds"] for r in ok])), 1) if ok else None,
        "cost_per_run": round(spent / max(1, len(ok)), 5) if ok else None}, indent=2))
    print(f"[{dkey}] {label} done: {len(ok)}/{len(rows)} ok | spend ${spent:.4f} "
          f"(${spent/max(1,len(ok)):.4f}/run)")
    return rows

# Load-check every subset record through the engine DataLoader (cheap, no spend).
from src.full_stack.backend.utils.core.data_loader import DataLoader
dl = DataLoader(); checked = 0
for dkey, d in DATASETS.items():
    for job in build_jobs(dkey, [d["subset_tier"]]):
        assert job["src_dir"].is_dir(), f"missing input dir: {job['src_dir']}"
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            data = dl.load(job["src_dir"])
        cov = data.data_overview
        assert cov.domain_coverage and cov.total_tokens and cov.total_tokens > 0, f"bad overview: {job['src_dir']}"
        blob = json.dumps(getattr(data.multimodal_data, "raw", None) or getattr(data.multimodal_data, "__dict__", {}), default=str)
        for tok in job["outputs"]:
            assert tok not in blob, f"target {tok} leaked into inputs: {job['src_dir']}"
        checked += 1
def _reg_frame(rows):
    "Tidy (subject, tier, output, predicted, truth, error) frame from a dataset's results."
    out = []
    for r in rows:
        if not r.get("ok"):
            continue
        for o in r["outputs"]:
            pred = r["prediction"]["regression"].get(o); gt = (r["ground_truth"] or {}).get(o)
            out.append(dict(subject=r["subject"], tier=r["tier"], output=o, predicted=pred, truth=gt,
                            error=None if (pred is None or gt is None) else pred - gt))
    return pd.DataFrame(out)

RUN = {}   # dataset key -> subset results (filled by the per-dataset cells below)
print(f"OK: {checked} full-tier subset records load cleanly through the engine, 0 target leaks.")

## 3. INTELLIGENCE subset run (tier T4, all self-report)

Runs the 2 diverse subjects on the all-self-report tier (demographics, personality, motivation/affect,
identity). The engine predicts total IST intelligence and, beneath it, the three subscales. The brain tiers
(T5/T6) are the full-complexity end of the ladder and are covered by the cost estimate in section 6.
Run this cell on its own to reproduce; it caches so a re-run does not re-spend.

In [ ]:
RUN["INTELLIGENCE"] = run_dataset("INTELLIGENCE", [DATASETS["INTELLIGENCE"]["subset_tier"]], rerun=False)

### 3.1 Recovered intelligence vs ground truth

In [ ]:
df = _reg_frame(RUN.get("INTELLIGENCE", []))
if not df.empty:
    outs = ["IST_intelligence_total", "IST_fluid", "IST_memory", "IST_crystallised"]
    fig, ax = plt.subplots(1, 1, figsize=(9, 5), constrained_layout=True)
    x = np.arange(len(outs)); w = 0.36
    for i, subj in enumerate(sorted(df["subject"].unique())):
        s = df[df["subject"] == subj].set_index("output")
        pr = [s.loc[o, "predicted"] if o in s.index else np.nan for o in outs]
        tr = [s.loc[o, "truth"] if o in s.index else np.nan for o in outs]
        ax.bar(x + (i - 0.5) * w, pr, w * 0.9, label=f"{subj} predicted", alpha=0.85)
        ax.scatter(x + (i - 0.5) * w, tr, color="k", zorder=3, marker="_", s=260,
                   label="ground truth" if i == 0 else None)
    ax.set_xticks(x); ax.set_xticklabels([o.replace("IST_", "") for o in outs])
    ax.set_ylabel("native IST points"); ax.set_title("INTELLIGENCE (T6): predicted bars vs true (black ticks)")
    ax.legend(fontsize=8, frameon=False); plt.show()
    print(df[["subject", "output", "predicted", "truth", "error"]].round(1).to_string(index=False))
else:
    print("Run the INTELLIGENCE subset cell above first.")

## 4. PSYCHOSIS subset run (tier T2, clinical profile)

Runs one case and one control on the clinical-profile tier (demographics, socio-economic status, MATRICS
cognition, WASI IQ, GAF/SFS functioning). The engine predicts the binary diagnosis, then BPRS total
severity, then the SAPS/SANS symptom globals beneath it. The full 836-feature EEG tier (T3) is the
full-complexity end of the ladder and is covered by the cost estimate in section 6.

In [ ]:
RUN["PSYCHOSIS"] = run_dataset("PSYCHOSIS", [DATASETS["PSYCHOSIS"]["subset_tier"]], rerun=False)

### 4.1 Recovered diagnosis and symptom severity

In [ ]:
rows = RUN.get("PSYCHOSIS", [])
if any(r.get("ok") for r in rows):
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.4), constrained_layout=True)
    labels, correct = [], []
    for r in rows:
        if not r.get("ok"):
            continue
        pred = r["prediction"]["label"]; true = r["diagnosis"]
        labels.append(f"{r['subject'][:18]}\ntrue: {true}"); correct.append(pred == true)
        ax[0].annotate(f"predicted:\n{pred}", (len(labels) - 1, 0.5), ha="center", va="center", fontsize=9)
    ax[0].bar(range(len(labels)), [1] * len(labels), color=["#59a14f" if c else "#e15759" for c in correct])
    ax[0].set_xticks(range(len(labels))); ax[0].set_xticklabels(labels, fontsize=8)
    ax[0].set_yticks([]); ax[0].set_title("Diagnosis (green=correct, red=wrong)")
    df = _reg_frame(rows)
    sym = df[df["output"] != "target__psychosis__case_control_label"].copy()
    sym["name"] = sym["output"].str.split("__").str[-1].str.slice(0, 16)
    for subj in sorted(sym["subject"].unique()):
        s = sym[sym["subject"] == subj]
        ax[1].scatter(s["truth"], s["predicted"], label=subj[:18], s=40)
    lim = [0, max(1, np.nanmax(sym[["truth", "predicted"]].to_numpy()) * 1.1)]
    ax[1].plot(lim, lim, "k--", lw=0.8, alpha=0.6); ax[1].set_xlim(lim); ax[1].set_ylim(lim)
    ax[1].set_xlabel("true score"); ax[1].set_ylabel("predicted score")
    ax[1].set_title("BPRS total + SAPS/SANS globals: pred vs true"); ax[1].legend(fontsize=8, frameon=False)
    plt.show()
    print(f"diagnosis correct: {sum(correct)}/{len(correct)}")
    print(df[["subject", "output", "predicted", "truth", "error"]].round(2).to_string(index=False))
else:
    print("Run the PSYCHOSIS subset cell above first.")

## 5. NUMERACY subset run (tier T3 coarse, both phenotypes)

Runs 2 subjects on the network-level lesion tier for BOTH numeracy phenotypes (approximate and precise),
so their dissociation can be read on the same population-Z scale. The full per-parcel tier (T3 fine) is the
full-complexity end of the ladder and is covered by the cost estimate in section 6.

In [ ]:
RUN["NUMERACY"] = run_dataset("NUMERACY", [DATASETS["NUMERACY"]["subset_tier"]], rerun=False)

### 5.1 Approximate vs precise numeracy recovery

In [ ]:
df = _reg_frame(RUN.get("NUMERACY", []))
if not df.empty:
    subjects = sorted(df["subject"].unique())
    fig, ax = plt.subplots(1, 1, figsize=(9, 5), constrained_layout=True)
    x = np.arange(len(subjects)); w = 0.2
    cmap = {"approximate_numeracy": "#4e79a7", "precise_numeracy": "#e15759"}
    for j, (o, c) in enumerate(cmap.items()):
        pr = [df[(df.subject == s) & (df.output == o)]["predicted"].mean() for s in subjects]
        tr = [df[(df.subject == s) & (df.output == o)]["truth"].mean() for s in subjects]
        ax.bar(x + (j - 0.5) * w * 2, pr, w * 1.8, color=c, alpha=0.85, label=f"{o.split('_')[0]} pred")
        ax.scatter(x + (j - 0.5) * w * 2, tr, color="k", marker="_", s=200, zorder=3,
                   label="ground truth" if j == 0 else None)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xticks(x); ax.set_xticklabels(subjects); ax.set_ylabel("population Z-score")
    ax.set_title("NUMERACY (T3 lesion-fine): predicted bars vs true (black ticks)")
    ax.legend(fontsize=8, frameon=False); plt.show()
    print(df[["subject", "output", "predicted", "truth", "error"]].round(3).to_string(index=False))
    mae = df.dropna(subset=["error"]).groupby("output")["error"].apply(lambda e: e.abs().mean())
    print("\nMean absolute error (Z units):"); print(mae.round(3).to_string())
else:
    print("Run the NUMERACY subset cell above first.")

## 6. Full-batch API cost and wall-time estimate (per dataset)

The subset above measured the real cost and duration of a run on each dataset's SUBSET tier. Other tiers
cost more or less roughly in proportion to their input size, so the full-batch estimate scales the measured
subset-tier cost by each tier's input-token ratio (a floor keeps the fixed per-run overhead honest). This
gives, per dataset, the estimated cost and wall time to run the ENTIRE cohort across the ENTIRE tier ladder
(including the big brain/EEG tiers) before you commit to it.

In [ ]:
def tier_tokens(dkey, tier):
    "Input token budget of one subject's record at a tier (0 if unavailable)."
    d = DATASETS[dkey]
    for task in d["tasks"]:
        for subj in d["subjects"]:
            p = d["src_dir"](tier, subj, task["name"]) / "data_overview.json"
            if p.exists():
                return json.loads(p.read_text()).get("total_tokens", 0)
    return 0

# A run's cost is a fixed per-run overhead (system prompts, orchestration, reasoning) plus a part that
# scales with the participant's input tokens. We anchor BOTH the overhead floor and the token slope to the
# real measured cost/run on the full tier (from each subset run's meta), so the estimate is calibrated, not
# guessed. FLOOR = a low-token tier still pays most of the fixed overhead.
FLOOR = 0.6
est_rows = []
for dkey, d in DATASETS.items():
    meta_path = SCRATCH / f"results_{dkey}_subset_meta.json"
    if not meta_path.exists():
        print(f"[{dkey}] no subset meta yet; run its subset cell to calibrate the estimate."); continue
    meta = json.loads(meta_path.read_text())
    cpr_full = meta.get("cost_per_run") or 0.0            # measured $/run on the full tier
    avg_s = meta.get("avg_seconds") or 0.0
    full_tokens = max(1, tier_tokens(dkey, d["subset_tier"]))  # anchor = the measured subset tier
    def scale(t):                                          # per-tier factor vs the full tier (token ratio, floored)
        return max(FLOOR, tier_tokens(dkey, t) / full_tokens)
    n_tasks = len(d["tasks"])
    total_cost = sum(d["n_cohort"] * n_tasks * cpr_full * scale(t) for t in d["tiers"])
    total_secs = sum(d["n_cohort"] * n_tasks * avg_s * scale(t) for t in d["tiers"])
    total_runs = d["n_cohort"] * n_tasks * len(d["tiers"])
    est_rows.append(dict(dataset=dkey, measured_usd_per_run=round(cpr_full, 4),
                         avg_seconds_per_run=round(avg_s, 0), full_batch_runs=int(total_runs),
                         est_cost_usd=round(total_cost, 2),
                         est_wall_hours=round(total_secs / MAX_WORKERS / 3600, 2)))
if est_rows:
    est = pd.DataFrame(est_rows)
    print("Full-batch estimate (entire cohort x entire tier ladder), calibrated on the measured subset:\n")
    print(est.to_string(index=False))
    print(f"\nTOTAL across all three datasets: ~${est['est_cost_usd'].sum():.2f}, "
          f"~{est['est_wall_hours'].sum():.1f} wall-hours at {MAX_WORKERS} workers "
          f"(each full-cohort run also needs all-subject compass_inputs built first).")
    print("Costs scale the MEASURED full-tier $/run by each tier's input-token ratio; the actual figure "
          "depends on how much the engine reasons per run.")

## 7. Full-batch runs (guarded, NOT run here)

Each cell below runs one dataset's ENTIRE cohort across its ENTIRE tier ladder. They are guarded by
`RUN_FULL_BATCH` so they never fire by accident; read the cost estimate above, set the flag to `True`, and
run one dataset at a time. Results cache to system temp under `results_<dataset>_full.json`. To evaluate
the whole cohort, first regenerate that dataset's compass_inputs for all subjects (see its pipeline).

In [ ]:
RUN_FULL_BATCH = False   # set True to actually spend on a full-cohort, full-ladder run

if RUN_FULL_BATCH:
    RUN["INTELLIGENCE_full"] = run_dataset("INTELLIGENCE", DATASETS["INTELLIGENCE"]["tiers"], label="full")
else:
    print("INTELLIGENCE full batch is guarded (RUN_FULL_BATCH=False). See the estimate in section 6.")

In [ ]:
if RUN_FULL_BATCH:
    RUN["PSYCHOSIS_full"] = run_dataset("PSYCHOSIS", DATASETS["PSYCHOSIS"]["tiers"], label="full")
else:
    print("PSYCHOSIS full batch is guarded (RUN_FULL_BATCH=False). See the estimate in section 6.")

In [ ]:
if RUN_FULL_BATCH:
    RUN["NUMERACY_full"] = run_dataset("NUMERACY", DATASETS["NUMERACY"]["tiers"], label="full")
else:
    print("NUMERACY full batch is guarded (RUN_FULL_BATCH=False). See the estimate in section 6.")

## 8. Summary

In [ ]:
allrows = [r for k in ("INTELLIGENCE", "PSYCHOSIS", "NUMERACY") for r in RUN.get(k, [])]
ok = [r for r in allrows if r.get("ok")]
print(f"subset engine runs: {len(allrows)} total | {len(ok)} ok | {len(allrows)-len(ok)} errored")
if ok:
    print(f"avg run time: {np.mean([r['seconds'] for r in ok]):.0f}s | model: {MODEL}")
    frames = [_reg_frame(RUN.get(k, [])) for k in ("INTELLIGENCE", "PSYCHOSIS", "NUMERACY")]
    allreg = pd.concat([f for f in frames if not f.empty], ignore_index=True) if any(not f.empty for f in frames) else pd.DataFrame()
    if not allreg.empty:
        allreg["abserr"] = allreg["error"].abs()
        print("\nmean absolute error by output:")
        print(allreg.dropna(subset=["abserr"]).groupby("output")["abserr"].mean().round(2).to_string())
for r in [r for r in allrows if not r.get("ok")][:8]:
    print(f"  ERROR {r['dataset']} {r['tier']} {r['subject']}: {r.get('error')}")
print("\nSubset validation complete. Full-cohort runs are estimated in section 6 and guarded in section 7.")